In [1]:
import pandas as pd

# 1. Cargar dataset
file_path = "dataset_ingles.csv"  # ajusta si es necesario
df = pd.read_csv(file_path, sep=';', encoding='utf-8')

# --- Normalización de nombres de columnas (opcional pero recomendable) ---
df.columns = df.columns.str.strip().str.upper()

# Aseguramos nombres esperados
# (ajusta si en tu dataset tienen otro nombre exacto)
col_fecha_nac = "DATANAIXEMENT"
col_fecha = "DATA"

# 2. Eliminar registros con fecha de nacimiento nula
df = df.dropna(subset=[col_fecha_nac])

# 3. Convertir a datetime
df[col_fecha_nac] = pd.to_datetime(df[col_fecha_nac], errors='coerce')
df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')

# Volvemos a eliminar posibles errores tras conversión
df = df.dropna(subset=[col_fecha_nac, col_fecha])

# 4. Crear campo EDAD
df["EDAD"] = (df[col_fecha] - df[col_fecha_nac]).dt.days // 365

# 5. Filtrar edades entre 8 y 70 años
df = df[(df["EDAD"] > 8) & (df["EDAD"] < 70)]

# 6. Filtrar registros que tengan las 5 materias
# Clave única: IDALUMNE;IDCURS;IDEVALUACIO
group_cols = ["IDALUMNE", "IDCURS", "IDEVALUACIO"]

# Contar número de materias por grupo
materias_por_grupo = df.groupby(group_cols)["MATERIA"].nunique().reset_index()

# Nos quedamos solo con los que tienen exactamente 5 materias
grupos_validos = materias_por_grupo[materias_por_grupo["MATERIA"] == 5]

# Hacemos merge para filtrar el dataframe original
df = df.merge(grupos_validos[group_cols], on=group_cols, how="inner")

# 7. Guardar resultado en CSV
output_path = "dataset_cleaned.csv"
df.to_csv(output_path, sep=';', index=False, encoding='utf-8')

print("Proceso completado.")
print(f"Archivo guardado en: {output_path}")
print(f"Número final de registros: {len(df)}")

C:\Users\Usuario\AppData\Local\Temp\ipykernel_99460\3761407314.py:19: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col_fecha_nac] = pd.to_datetime(df[col_fecha_nac], errors='coerce')
C:\Users\Usuario\AppData\Local\Temp\ipykernel_99460\3761407314.py:20: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')


Proceso completado.
Archivo guardado en: dataset_cleaned.csv
Número final de registros: 23070
